# 06 — Ferrari Championship Lead Conversion

Full version of the Ferrari narrative: of all seasons from 2014-2025 where Ferrari
held the constructor championship lead at mid-season, how many did they convert to a title?

Comparison constructors: Mercedes, Red Bull.

Data: 2014-2021 via Jolpica API; 2022-2025 via FastF1 (championship_trajectory parquet).

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd

import scripts.export_data as exp
from src.data.jolpica_client import get_constructor_standings_by_round
from src.analysis.championship_lead_conversion import mid_season_leaders, conversion_rate_summary

## 1. Fetch 2014-2021 standings from Jolpica

Each season is cached after first fetch. Expect ~2 minutes on first run, instant thereafter.

In [2]:
HYBRID_SEASONS = list(range(2014, 2022))

jolpica_frames = []
for season in HYBRID_SEASONS:
    df = get_constructor_standings_by_round(season)
    jolpica_frames.append(df)
    n_rounds = df['round'].nunique()
    n_cons = df['constructor_id'].nunique()
    print(f'{season}: {n_rounds} rounds, {n_cons} constructors')

standings_hybrid = pd.concat(jolpica_frames, ignore_index=True)
print('Hybrid Era total:', len(standings_hybrid), 'rows')

2014: 19 rounds, 11 constructors
2015: 19 rounds, 10 constructors
2016: 21 rounds, 11 constructors
2017: 20 rounds, 10 constructors
2018: 21 rounds, 10 constructors
2019: 21 rounds, 10 constructors
2020: 17 rounds, 10 constructors
2021: 22 rounds, 10 constructors
Hybrid Era total: 1639 rows


## 2. Load 2022-2025 from championship_trajectory parquet

In [3]:
traj = exp.read("championship_trajectory")
standings_ground = (
    traj[traj["season"].isin(range(2022, 2026))]
    .rename(columns={"cumulative_points": "points"})
    [["season", "round", "constructor_id", "constructor_name", "points"]]
    .copy()
)
for s in sorted(standings_ground["season"].unique()):
    d = standings_ground[standings_ground["season"]==s]
    n_r = d["round"].nunique()
    n_c = d["constructor_id"].nunique()
    print(f"{s}: {n_r} rounds, {n_c} constructors")

2022: 22 rounds, 10 constructors
2023: 22 rounds, 10 constructors
2024: 24 rounds, 10 constructors
2025: 24 rounds, 10 constructors


## 3. Combine and compute mid-season leaders

In [4]:
all_standings = pd.concat([standings_hybrid, standings_ground], ignore_index=True)

lead_df = mid_season_leaders(all_standings, min_round=4)
n_rows = len(lead_df)
n_seasons = lead_df["season"].nunique()
print(f"Mid-season leaders table: {n_rows} rows ({n_seasons} seasons)")
print()
print(lead_df[["season","constructor_id","first_led_round","won_title","final_position"]].to_string(index=False))

Mid-season leaders table: 16 rows (12 seasons)

 season constructor_id  first_led_round  won_title  final_position
   2014       mercedes                4       True               1
   2015       mercedes                4       True               1
   2016       mercedes                4       True               1
   2017       mercedes                4       True               1
   2017        ferrari                6      False               2
   2018        ferrari                4      False               2
   2018       mercedes                5       True               1
   2019       mercedes                4       True               1
   2020       mercedes                4       True               1
   2021       mercedes                4       True               1
   2021       red_bull                5      False               2
   2022        ferrari                4      False               2
   2022       red_bull                6       True               1
   2023       

## 4. Conversion rate summary

In [5]:
FOCUS = ['ferrari', 'mercedes', 'red_bull']

summary = conversion_rate_summary(lead_df, constructors=FOCUS)
print(summary[['constructor_name','seasons_led','titles_won','conversion_rate','seasons_list']].to_string(index=False))

constructor_name  seasons_led  titles_won  conversion_rate                                   seasons_list
        Mercedes            8           8              1.0 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021
 Red Bull Racing            4           2              0.5                         2021, 2022, 2023, 2024
         Ferrari            3           0              0.0                               2017, 2018, 2022


## 5. Export

In [6]:
exp.export(ferrari_lead_conversion=lead_df)
print("Export complete")

  Exporting ferrari_lead_conversion (16 rows)... done.

manifest.json updated (1 file(s) exported).
Export complete
